# Hybrid Time Embedding — Fine-Tuning Notebook
6-layer Qwen2.5-7B architecture for Date Arithmetic and Duration QA.
Run cells top-to-bottom. Set `config.data_dir` before Section 1.

## Section 0: Setup & Config

In [ ]:
# Cell 0.1 — Imports
import os, sys, random, json, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path(".").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version:    {torch.version.cuda}")
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM:            {vram:.1f} GB")

In [ ]:
# Cell 0.2 — HybridConfig
from hybrid_time_embedding.src.utils.config import HybridConfig

config = HybridConfig(
    # Model
    base_model_name="Qwen/Qwen2.5-7B",
    d_model=3584,
    n_learned_freq=8,
    n_random_freq=16,
    gate_init=0.1,
    gate_threshold=0.05,

    # LoRA
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    lora_target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj"],

    # Loss weights
    lambda_torus=0.3,
    lambda_consist=0.5,
    lambda_gate=1.0,

    # Phase 1
    phase1_epochs=2,
    phase1_lr_emb=1e-3,
    phase1_batch_size=16,

    # Phase 2
    phase2_epochs=3,
    phase2_lr_backbone=2e-5,
    phase2_lr_emb=1e-4,
    phase2_lr_heads=1e-3,
    phase2_batch_size=8,
    phase2_grad_accum=8,

    # Phase 3
    phase3_lr=5e-7,
    phase3_n_generations=8,
    phase3_beta=0.04,
    phase3_freeze_emb_steps=500,

    # Paths
    data_dir="./data",
    output_dir="./models",
    experiment_dir="./experiments",
    log_dir="./experiments/logs",
)
print("Config loaded:", config)

In [ ]:
# Cell 0.3 — Environment & paths  (mirrors run_finetune_phase2_colab.ipynb SETUP 2–3)
import os, sys
from pathlib import Path

USE_DRIVE = False  # True → mount Drive to persist checkpoints + outputs

# ── Colab detection ───────────────────────────────────────────────────────────
def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IN_COLAB = _in_colab()

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# ── REPO_DIR — fixed path on Colab (mirrors phase2 SETUP 2), one level up locally ──
if IN_COLAB:
    REPO_DIR = '/content/Temporal_Reasoning'
    if not os.path.isdir(REPO_DIR):
        REPO_URL = 'https://github.com/<YOUR_USER>/Temporal_Reasoning.git'  # TODO: set your repo
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
    else:
        os.system(f'cd {REPO_DIR} && git pull')
else:
    # Notebook is at REPO_DIR/hybrid_time_embedding/finetune.ipynb → one parent up
    REPO_DIR = str(Path(os.getcwd()).resolve().parent)

# chdir so registry's relative paths (Dataset/Raw/...) resolve correctly
os.chdir(REPO_DIR)
print('CWD / REPO_DIR:', os.getcwd())

# ── Dataset / output paths ────────────────────────────────────────────────────
if USE_DRIVE and IN_COLAB:
    DRIVE_ROOT   = '/content/drive/MyDrive/Temporal_Reasoning'
    DATASET_ROOT = os.path.join(DRIVE_ROOT, 'Dataset')
    DRIVE_OUT    = os.path.join(DRIVE_ROOT, 'outputs_hybrid')
    local_ds = os.path.join(REPO_DIR, 'Dataset')
    if not os.path.exists(local_ds):
        os.symlink(DATASET_ROOT, local_ds)
    print('Dataset ->', os.readlink(local_ds) if os.path.islink(local_ds) else local_ds)
else:
    DATASET_ROOT = os.path.join(REPO_DIR, 'Dataset')
    DRIVE_OUT    = os.path.join(REPO_DIR, 'outputs_hybrid')
    if not os.path.exists(DATASET_ROOT):
        raise FileNotFoundError(f'Dataset not found at {DATASET_ROOT}')
    print('Dataset ->', DATASET_ROOT)

# ── Checkpoint dir (local session by default; on Drive when USE_DRIVE=True) ───
CHECKPOINT_DIR = (
    os.path.join(DRIVE_ROOT, 'checkpoints/hybrid_time_emb') if (USE_DRIVE and IN_COLAB)
    else '/content/checkpoints/hybrid_time_emb' if IN_COLAB
    else os.path.join(REPO_DIR, 'checkpoints/hybrid_time_emb')
)

os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Override config paths (mirrors phase2's raw['output_dir'] = CHECKPOINT_DIR)
config.output_dir     = CHECKPOINT_DIR
config.experiment_dir = DRIVE_OUT
config.log_dir        = os.path.join(DRIVE_OUT, 'logs')

print('USE_DRIVE      :', USE_DRIVE)
print('DRIVE_OUT      :', DRIVE_OUT)
print('CHECKPOINT_DIR :', CHECKPOINT_DIR)

## Section 1: Data Loading & Inspection

In [ ]:
# Cell 1.1 — Load datasets via registry  (mirrors run_finetune_phase2_colab.ipynb data loading)
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from src.data.registry import load_dataset as registry_load
from hybrid_time_embedding.src.data.dataset import TemporalQADataset
from hybrid_time_embedding.src.data.collator import DataCollatorWithTimestamps
import random

# ── Pool config — mirrors SFTRunConfig parameters from phase2 notebook ────────
REGISTRY_DATASETS = ['bigbench_date', 'udst_duration', 'vlsp_date', 'vlsp_duration']
TRAIN_POOL_START  = 0      # rows to skip from front (set >0 to avoid eval-set overlap)
TRAIN_POOL_SIZE   = None   # None → use all rows after TRAIN_POOL_START
VAL_RATIO         = 0.1    # fraction for val (and test) — matches SFTRunConfig default
SHUFFLE_SEED      = 42

# ── Load & slice pool ─────────────────────────────────────────────────────────
pool = []
for ds_name in REGISTRY_DATASETS:
    samples = registry_load(ds_name)
    print(f'  {ds_name}: {len(samples)} samples')
    pool.extend(samples)

pool = pool[TRAIN_POOL_START:]
if TRAIN_POOL_SIZE is not None:
    pool = pool[:TRAIN_POOL_SIZE]

random.seed(SHUFFLE_SEED)
random.shuffle(pool)
print(f'Pool after slice: {len(pool):,} samples')

# ── Train / val / test split ──────────────────────────────────────────────────
n      = len(pool)
n_val  = max(1, int(n * VAL_RATIO))
n_test = n_val
print(f'pool={n:,}  →  train={n - 2*n_val:,} | val={n_val:,} | test={n_test:,}')

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name, trust_remote_code=True)
tokenizer.add_special_tokens({'additional_special_tokens': ['[TIME_START]', '[TIME_END]']})
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── TemporalQADataset (from_samples converts registry Sample → training format) ─
train_ds = TemporalQADataset.from_samples(pool[:n - 2*n_val])
val_ds   = TemporalQADataset.from_samples(pool[n - 2*n_val : n - n_val])
test_ds  = TemporalQADataset.from_samples(pool[n - n_val:])

collator = DataCollatorWithTimestamps(tokenizer)

train_loader = DataLoader(train_ds, batch_size=config.phase1_batch_size,
                          shuffle=True, collate_fn=collator,
                          num_workers=config.dataloader_num_workers)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, collate_fn=collator)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, collate_fn=collator)

print(f'train: {len(train_ds):,}  val: {len(val_ds):,}  test: {len(test_ds):,}')
print('Sample:', train_ds[0])

In [ ]:
# Cell 1.1 — Load datasets
from torch.utils.data import DataLoader
from hybrid_time_embedding.src.data.dataset import TemporalQADataset
from hybrid_time_embedding.src.data.collator import DataCollatorWithTimestamps
from transformers import AutoTokenizer

# NOTE: set config.data_dir to your actual data directory before running
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name, trust_remote_code=True)
tokenizer.add_special_tokens({"additional_special_tokens": ["[TIME_START]", "[TIME_END]"]})
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

collator = DataCollatorWithTimestamps(tokenizer)

train_ds = TemporalQADataset(config.data_dir, split="train")
val_ds   = TemporalQADataset(config.data_dir, split="val")
test_ds  = TemporalQADataset(config.data_dir, split="test")

train_loader = DataLoader(train_ds, batch_size=config.phase1_batch_size,
                          shuffle=True, collate_fn=collator,
                          num_workers=config.dataloader_num_workers)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collator)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collator)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")
print("Sample:", train_ds[0])

In [ ]:
# Cell 1.2 — Data statistics
arith_items  = [x for x in train_ds if x['subtask'] == 'date_arithmetic']
dur_items    = [x for x in train_ds if x['subtask'] == 'date_duration']
print(f"Arithmetic: {len(arith_items)} | Duration: {len(dur_items)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Timestamp distribution
ts = [x['timestamp'] * 2100 for x in train_ds]
axes[0].hist(ts, bins=50, color='steelblue')
axes[0].set_title('Timestamp Distribution (years)')
axes[0].set_xlabel('Year')

# Duration distribution
durs = [x['dur_label'] * 2100 for x in dur_items]
axes[1].hist(durs, bins=50, color='coral')
axes[1].set_title('Duration Distribution (years)')
axes[1].set_xlabel('Duration')

# Arithmetic offset distribution
arith_ans = [x['arith_label'] * 2100 for x in arith_items]
axes[2].hist(arith_ans, bins=50, color='forestgreen')
axes[2].set_title('Arithmetic Answer Distribution')
axes[2].set_xlabel('Year')

plt.tight_layout()
plt.show()

## Section 2: Model Initialization

In [ ]:
# Cell 2.1 — Load base model
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

backbone = AutoModelForCausalLM.from_pretrained(
    config.base_model_name,
    torch_dtype=torch.bfloat16,
    quantization_config=quant_config,
    trust_remote_code=True,
)
backbone.resize_token_embeddings(len(tokenizer))
print(backbone)

In [ ]:
# Cell 2.2 — Apply LoRA
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    target_modules=config.lora_target_modules,
    lora_dropout=config.lora_dropout,
    bias="none",
    task_type="FEATURE_EXTRACTION",
)

total_params_before = sum(p.numel() for p in backbone.parameters())
backbone = get_peft_model(backbone, lora_config)
trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total_params_after = sum(p.numel() for p in backbone.parameters())
pct = 100 * trainable_params / total_params_after
print(f"Total params:    {total_params_after:,}")
print(f"Trainable:       {trainable_params:,} ({pct:.2f}% of total)")

In [ ]:
# Cell 2.3 — Attach custom layers and verify forward pass
from hybrid_time_embedding.src.models.full_model import HybridTemporalModel

model = HybridTemporalModel(config=config, backbone=backbone, tokenizer=tokenizer)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Dummy inputs
dummy_ids  = torch.randint(0, len(tokenizer), (2, 32)).to(device)
dummy_mask = torch.ones(2, 32, dtype=torch.long).to(device)
dummy_ts   = torch.rand(2).to(device)

# Shape check (no grad needed)
with torch.no_grad():
    arith_out, dur_out, gate_reg = model(dummy_ids, dummy_mask, dummy_ts)

print(f"arith_pred shape: {arith_out.shape}  (expected [2,1])")
print(f"dur_pred shape:   {dur_out.shape}    (expected [2,1])")
print(f"gate_reg:         {gate_reg.item():.4f}")

# Gradient flow check — separate forward pass outside no_grad
model.zero_grad()
arith_out, dur_out, gate_reg = model(dummy_ids, dummy_mask, dummy_ts)
loss_test = arith_out.sum() + dur_out.sum()
loss_test.backward()
for name, p in model.named_parameters():
    if p.requires_grad and p.grad is not None:
        print(f"  grad OK: {name[:60]}")
        break
print("Gradient flow verified.")

In [ ]:
# Cell 2.4 — Parameter groups summary
from tabulate import tabulate

components = {
    "Token Embedding (frozen)": model.backbone.get_input_embeddings(),
    "HybridTimeEmbedding":      model.time_embedding,
    "OptimalFusion":            model.fusion,
    "AttentionPooling":         model.pooler,
    "ArithmeticHead":           model.arith_head,
    "DurationHead":             model.dur_head,
}

rows = []
for name, comp in components.items():
    total  = sum(p.numel() for p in comp.parameters())
    trainable = sum(p.numel() for p in comp.parameters() if p.requires_grad)
    rows.append([name, f"{total:,}", "Yes" if trainable > 0 else "No", "varies"])

lora_total = sum(p.numel() for n, p in model.backbone.named_parameters() if "lora_" in n)
rows.append(["LoRA Adapters (L4-27)", f"{lora_total:,}", "Yes", "2e-5"])

print(tabulate(rows, headers=["Component", "#Params", "Trainable", "LR"], tablefmt="grid"))

## Section 3: Phase 1 — Embedding Warmup

In [ ]:
# Cell 3.1 — Setup Phase 1
from hybrid_time_embedding.src.utils.logging_utils import setup_logging
from hybrid_time_embedding.src.training.callbacks import SmartCheckpointSaver
from hybrid_time_embedding.src.training.trainer import PhaseAwareTrainer

os.makedirs(config.log_dir, exist_ok=True)
os.makedirs(config.output_dir, exist_ok=True)

logger, tb_writer, wandb_run = setup_logging(config.log_dir, experiment_name="hybrid_time_emb")

checkpoint_saver = SmartCheckpointSaver(
    output_dir=config.output_dir,
    top_k=config.checkpoint_top_k,
    improvement_threshold=config.checkpoint_improvement_threshold,
    save_every_steps=config.checkpoint_save_every_steps,
)

trainer = PhaseAwareTrainer(
    model=model,
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
    checkpoint_saver=checkpoint_saver,
    logger=logger,
    tb_writer=tb_writer,
    wandb_run=wandb_run,
)

print("Phase 1 trainer ready.")

In [ ]:
# Cell 3.2 — Phase 1 Training Loop
phase1_metrics = trainer.train_phase1()
print("Phase 1 complete:", phase1_metrics)

In [ ]:
# Cell 3.3 — Phase 1 Results
# (TensorBoard: run `tensorboard --logdir experiments/logs/tb` in terminal)
print("Best checkpoint path:", checkpoint_saver.output_dir)
print("Phase 1 val metrics:")
for k, v in phase1_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

## Section 4: Phase 2 — Supervised Fine-Tuning

In [ ]:
# Cell 4.1 — Setup Phase 2
# Swap to phase2 batch size loader
train_loader_p2 = DataLoader(
    train_ds, batch_size=config.phase2_batch_size,
    shuffle=True, collate_fn=collator,
    num_workers=config.dataloader_num_workers
)
trainer.train_loader = train_loader_p2
print("Phase 2 setup: LoRA layers will unfreeze, backbone layers 0-3 stay frozen.")

In [ ]:
# Cell 4.2 — Phase 2 Training Loop
phase2_metrics = trainer.train_phase2()
print("Phase 2 complete:", phase2_metrics)

In [ ]:
# Cell 4.3 — Phase 2 Results
print("Phase 2 full evaluation:")
for k, v in phase2_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

print(f"\nGate value after Phase 2: {model.gate_value:.4f}")
print("Learned frequencies:",
      [f"{f:.4f}" for f in model.time_embedding.log_freq_learned.exp().detach().cpu().tolist()])

## Section 5: Phase 3 — GRPO Reinforcement Learning

In [ ]:
# Cell 5.1 — Setup Phase 3
print("Phase 3: GRPO RL.")
print(f"  HybridTimeEmb frozen for first {config.phase3_freeze_emb_steps} steps.")
print(f"  N generations: {config.phase3_n_generations}, beta: {config.phase3_beta}, lr: {config.phase3_lr}")

In [ ]:
# Cell 5.2 — Phase 3 Training Loop
phase3_metrics = trainer.train_phase3_grpo()
print("Phase 3 complete:", phase3_metrics)

In [ ]:
# Cell 5.3 — Phase 3 Results: compare Phase 2 vs Phase 3
print("Phase 2 vs Phase 3 comparison:")
headers = ["Metric", "Phase 2", "Phase 3"]
rows = []
for key in ["val/mae_overall", "val/exact_match_arithmetic", "val/consistency_rate"]:
    rows.append([key, f"{phase2_metrics.get(key, 0):.4f}", f"{phase3_metrics.get(key, 0):.4f}"])
from tabulate import tabulate
print(tabulate(rows, headers=headers, tablefmt="grid"))

## Section 6: Evaluation — Zero-Shot & Few-Shot

8 experiments: 4 datasets × {zero_shot, few_shot (k=3)}. Each experiment in its own cell — mirrors `PotCot.ipynb`.

- **Zero-shot**: trained model evaluated directly, no in-context examples.
- **Few-shot (k=3)**: 3 training examples tokenized and prepended to every input; tests whether the Qwen3-8B backbone leverages in-context demonstrations at inference time.

Results saved to `DRIVE_OUT/<method>/<dataset>/` (predictions.jsonl + metrics.json).  
Note: binary yes/no duration rows are skipped by `TemporalQADataset.from_samples`.

In [ ]:
# Cell 6.1 — Load best checkpoint
best_metrics = checkpoint_saver.load_best(model)
print("Loaded best checkpoint. Val MAE:", best_metrics.get("val_mae", "N/A"))

In [ ]:
# Cell 6.0 — eval helper  (run once before EXP 1–8)
import json, time, random
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from src.data.registry import load_dataset as registry_load
from hybrid_time_embedding.src.data.dataset import TemporalQADataset
from hybrid_time_embedding.src.data.collator import DataCollatorWithTimestamps
from hybrid_time_embedding.src.utils.metrics import compute_metrics

EXP_RESULTS = {}   # keyed by "method/dataset", collected for the summary table

def run_eval_exp(exp_id, ds_name, *, method='zero_shot', k_shot=3,
                 verbose=True, verbose_first_n=5, verbose_every=100,
                 running_score_every=50, output_dir=DRIVE_OUT):
    """
    Evaluate trained HybridTemporalModel on one dataset.
    Mirrors run_exp() from PotCot.ipynb.

    method: 'zero_shot' | 'few_shot'
    Few-shot prepends k_shot training examples as a token prefix to every batch
    item (prepend → truncate right side to 512 tokens).
    """
    print(f"\n{'─'*62}")
    print(f"  EXP {exp_id}  {method} × {ds_name}")
    print(f"{'─'*62}")

    # ── Split (same seed/ratio as training to avoid leakage) ──────────────────
    all_samples = registry_load(ds_name)
    rng = random.Random(42)
    shuffled = list(all_samples); rng.shuffle(shuffled)
    n_test  = max(1, int(len(shuffled) * 0.1))
    test_samples = shuffled[-n_test:]
    shot_samples = shuffled[:k_shot] if method == 'few_shot' else []
    print(f"  registry={len(all_samples):,}  test={n_test:,}  shots={len(shot_samples)}")

    # ── DataLoader ────────────────────────────────────────────────────────────
    test_ds_exp = TemporalQADataset.from_samples(test_samples)
    n_valid = len(test_ds_exp)
    print(f"  valid (non-binary) samples: {n_valid:,}")
    loader = DataLoader(test_ds_exp, batch_size=16, shuffle=False,
                        collate_fn=DataCollatorWithTimestamps(tokenizer))

    # ── Few-shot prefix tokens ─────────────────────────────────────────────────
    prefix_ids, prefix_mask = None, None
    if shot_samples:
        shot_text = '\n\n'.join(
            f"Q: {s.get('question','')}\nContext: {(s.get('context') or '').strip()}\nA: {s.get('gold','')}"
            for s in shot_samples
        ) + '\n\n'
        enc_p = tokenizer(shot_text, return_tensors='pt',
                          truncation=True, max_length=256, add_special_tokens=False)
        prefix_ids  = enc_p['input_ids'].to(device)       # [1, P]
        prefix_mask = enc_p['attention_mask'].to(device)

    # ── Eval loop ─────────────────────────────────────────────────────────────
    model.eval()
    all_arith_pred, all_arith_true = [], []
    all_dur_pred,   all_dur_true   = [], []
    all_start_times = []
    latencies = []
    sample_idx = 0

    for batch in loader:
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                 for k, v in batch.items()}

        ids  = batch['input_ids']
        mask = batch['attention_mask']
        if prefix_ids is not None:
            B    = ids.shape[0]
            ids  = torch.cat([prefix_ids.expand(B, -1),  ids],  dim=1)[:, -512:]
            mask = torch.cat([prefix_mask.expand(B, -1), mask], dim=1)[:, -512:]

        t0 = time.perf_counter()
        with torch.no_grad():
            arith_pred, dur_pred, _ = model(ids, mask, batch['timestamps'])
        latencies.append((time.perf_counter() - t0) * 1000)

        a_preds = arith_pred.squeeze(-1).cpu().tolist()
        d_preds = dur_pred.squeeze(-1).cpu().tolist()
        a_true  = batch['arith_labels'].cpu().tolist()
        d_true  = batch['dur_labels'].cpu().tolist()
        s_times = batch['start_times'].cpu().tolist()
        if not isinstance(a_preds, list):
            a_preds, d_preds = [a_preds], [d_preds]
            a_true, d_true, s_times = [a_true], [d_true], [s_times]

        for j in range(len(a_preds)):
            if verbose and sample_idx < verbose_first_n:
                print(f"  [{sample_idx+1}] arith pred={a_preds[j]*2100:.1f} gold={a_true[j]*2100:.1f} | "
                      f"dur pred={d_preds[j]*2100:.1f} gold={d_true[j]*2100:.1f}")
            sample_idx += 1

        all_arith_pred.extend(a_preds); all_arith_true.extend(a_true)
        all_dur_pred.extend(d_preds);   all_dur_true.extend(d_true)
        all_start_times.extend(s_times)

        if verbose and sample_idx > 0 and sample_idx % running_score_every == 0:
            mae_a = sum(abs(p-t) for p,t in zip(all_arith_pred, all_arith_true)) / len(all_arith_pred)
            mae_d = sum(abs(p-t) for p,t in zip(all_dur_pred,   all_dur_true))   / len(all_dur_pred)
            print(f"  [{sample_idx}/{n_valid}] running MAE — arith={mae_a*2100:.1f}yr  dur={mae_d*2100:.1f}yr")

    # ── Metrics ───────────────────────────────────────────────────────────────
    arith_m = compute_metrics(all_arith_pred, all_arith_true, subtask='arithmetic')
    dur_m   = compute_metrics(all_dur_pred,   all_dur_true, all_start_times, subtask='duration')

    metrics = {
        'exp_id': exp_id, 'dataset': ds_name, 'method': method, 'n': n_valid,
        'mae_arithmetic':         arith_m['mae'],
        'mae_duration':           dur_m['mae'],
        'mae_overall':            (arith_m['mae'] + dur_m['mae']) / 2,
        'exact_match_arithmetic': arith_m['exact_match'],
        'exact_match_duration':   dur_m['exact_match'],
        'within_1yr': (arith_m['within_1yr'] + dur_m['within_1yr']) / 2,
        'within_5yr': (arith_m['within_5yr'] + dur_m['within_5yr']) / 2,
        'avg_latency_ms': sum(latencies) / len(latencies) if latencies else 0,
    }

    print(f"\n── Results ──────────────────────────────────────────────────")
    print(f"  MAE   arith={metrics['mae_arithmetic']:.4f}  dur={metrics['mae_duration']:.4f}  "
          f"overall={metrics['mae_overall']:.4f}")
    print(f"  Exact arith={metrics['exact_match_arithmetic']:.3f}  dur={metrics['exact_match_duration']:.3f}")
    print(f"  Within-1yr={metrics['within_1yr']:.3f}  within-5yr={metrics['within_5yr']:.3f}")
    print(f"  n={n_valid}  avg_latency={metrics['avg_latency_ms']:.0f}ms")

    # ── Save ──────────────────────────────────────────────────────────────────
    if output_dir:
        out = Path(output_dir) / method / ds_name
        out.mkdir(parents=True, exist_ok=True)
        with open(out / 'predictions.jsonl', 'w') as f:
            for ap, dp, at, dt in zip(all_arith_pred, all_dur_pred,
                                       all_arith_true, all_dur_true):
                f.write(json.dumps({'arith_pred': ap, 'dur_pred': dp,
                                    'arith_true': at, 'dur_true': dt}) + '\n')
        with open(out / 'metrics.json', 'w') as f:
            json.dump(metrics, f, indent=2)
        print(f"\n── Saved outputs ────────────────────────────────────────────")
        print(f"  predictions ({n_valid} rows): {out}/predictions.jsonl")
        print(f"  metrics              : {out}/metrics.json")

    return metrics

print("run_eval_exp ready — run EXP 1–8 below.")

### Zero-shot

Direct evaluation — no in-context examples. Baseline performance of the trained model on each dataset.  
Rerun cells independently — each writes to `DRIVE_OUT/zero_shot/<dataset>/`.

In [ ]:
# === EXP 1/8 — zero_shot × UDST DurationQA (EN, Duration) ===
m = run_eval_exp('1/8', 'udst_duration', method='zero_shot',
                 verbose=True, verbose_every=50)
EXP_RESULTS['zero_shot/udst_duration'] = m
print(m)

In [ ]:
# === EXP 2/8 — zero_shot × BigBench DateUnderstanding (EN, DateArith) ===
m = run_eval_exp('2/8', 'bigbench_date', method='zero_shot',
                 verbose=True, verbose_every=50, running_score_every=50)
EXP_RESULTS['zero_shot/bigbench_date'] = m
print(m)

In [ ]:
# === EXP 3/8 — zero_shot × VLSP ViTempQA DateArith (VI, DateArith) ===
m = run_eval_exp('3/8', 'vlsp_date', method='zero_shot',
                 verbose=True, verbose_every=200)
EXP_RESULTS['zero_shot/vlsp_date'] = m
print(m)

In [ ]:
# === EXP 4/8 — zero_shot × VLSP ViTempQA DurationQA (VI, Duration) ===
m = run_eval_exp('4/8', 'vlsp_duration', method='zero_shot',
                 verbose=True, verbose_every=200)
EXP_RESULTS['zero_shot/vlsp_duration'] = m
print(m)

### Few-shot (k=3)

3 training examples tokenized and prepended to each input. Tests whether the Qwen3-8B backbone leverages in-context demonstrations at inference time.  
Rerun cells independently — each writes to `DRIVE_OUT/few_shot/<dataset>/`.

In [ ]:
# === EXP 5/8 — few_shot k=3 × UDST DurationQA (EN, Duration) ===
m = run_eval_exp('5/8', 'udst_duration', method='few_shot', k_shot=3,
                 verbose=True, verbose_every=50)
EXP_RESULTS['few_shot/udst_duration'] = m
print(m)

In [ ]:
# === EXP 6/8 — few_shot k=3 × BigBench DateUnderstanding (EN, DateArith) ===
m = run_eval_exp('6/8', 'bigbench_date', method='few_shot', k_shot=3,
                 verbose=True, verbose_every=50, running_score_every=50)
EXP_RESULTS['few_shot/bigbench_date'] = m
print(m)

In [ ]:
# === EXP 7/8 — few_shot k=3 × VLSP ViTempQA DateArith (VI, DateArith) ===
m = run_eval_exp('7/8', 'vlsp_date', method='few_shot', k_shot=3,
                 verbose=True, verbose_every=200)
EXP_RESULTS['few_shot/vlsp_date'] = m
print(m)

In [ ]:
# === EXP 8/8 — few_shot k=3 × VLSP ViTempQA DurationQA (VI, Duration) ===
m = run_eval_exp('8/8', 'vlsp_duration', method='few_shot', k_shot=3,
                 verbose=True, verbose_every=200)
EXP_RESULTS['few_shot/vlsp_duration'] = m
print(m)

In [ ]:
# Cell 6.9 — Summary table (run after EXP 1–8)
import pandas as pd
from tabulate import tabulate

rows = []
for key, m in EXP_RESULTS.items():
    rows.append({
        'EXP':       m['exp_id'],
        'Method':    m['method'],
        'Dataset':   m['dataset'],
        'MAE Overall': f"{m['mae_overall']:.4f}",
        'MAE Arith':   f"{m['mae_arithmetic']:.4f}",
        'MAE Dur':     f"{m['mae_duration']:.4f}",
        'Exact Arith': f"{m['exact_match_arithmetic']:.3f}",
        'Exact Dur':   f"{m['exact_match_duration']:.3f}",
        'Within-1yr':  f"{m['within_1yr']:.3f}",
        'n':           m['n'],
    })

df = pd.DataFrame(rows)
print(tabulate(df, headers='keys', tablefmt='grid', showindex=False))

csv_path = Path(DRIVE_OUT) / 'summary_hybrid.csv'
df.to_csv(csv_path, index=False)
print(f"\nSaved → {csv_path}")

In [ ]:
# Cell 6.10 — Zero-shot vs Few-shot comparison
from tabulate import tabulate

datasets = ['udst_duration', 'bigbench_date', 'vlsp_date', 'vlsp_duration']
rows = []
for ds in datasets:
    zs = EXP_RESULTS.get(f'zero_shot/{ds}', {})
    fs = EXP_RESULTS.get(f'few_shot/{ds}', {})
    if not zs or not fs:
        continue
    delta = fs['mae_overall'] - zs['mae_overall']
    rows.append([
        ds,
        f"{zs['mae_overall']:.4f}",
        f"{fs['mae_overall']:.4f}",
        f"{delta:+.4f}",
        '↓ better' if delta < -0.001 else '↑ worse' if delta > 0.001 else '≈ same',
    ])

print(tabulate(rows,
    headers=['Dataset', 'MAE zero-shot', 'MAE few-shot', 'Δ (fs−zs)', 'Direction'],
    tablefmt='grid'))

## Section 7: Inference Demo

In [ ]:
# Cell 7.1 — Single query demo
from hybrid_time_embedding.src.data.preprocessing import build_input_text, normalize_timestamp, extract_timestamps

example_queries = [
    {"query": "WW2 started in 1939 and lasted 6 years. When did it end?",
     "context": "World War II began September 1939.", "answer": 1945.0, "subtask": "arithmetic"},
    {"query": "How long did the Cold War last if it started in 1947 and ended in 1991?",
     "context": "The Cold War was a geopolitical period.", "answer": 44.0, "subtask": "duration"},
    {"query": "The Renaissance started in 1400 and lasted 200 years. When did it end?",
     "context": "The Renaissance was a cultural movement.", "answer": 1600.0, "subtask": "arithmetic"},
    {"query": "How long from 1776 to 1865?",
     "context": "American independence was declared in 1776.", "answer": 89.0, "subtask": "duration"},
    {"query": "The French Revolution began in 1789 and lasted 10 years. End year?",
     "context": "The Revolution transformed France.", "answer": 1799.0, "subtask": "arithmetic"},
]

print(f"{'Query':<60} {'Pred':>8} {'Truth':>8} {'Error':>8}")
print("-" * 90)
for ex in example_queries:
    text = build_input_text(ex["query"], ex["context"])
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(device) for k, v in enc.items()}
    ts_val = extract_timestamps(f"{ex['context']} {ex['query']}")
    ts_norm = torch.tensor([normalize_timestamp(ts_val[0] if ts_val else 2000.0)], device=device)
    with torch.no_grad():
        a_pred, d_pred, _ = model(enc["input_ids"], enc["attention_mask"], ts_norm)
    pred_val = a_pred.item() * 2100 if ex["subtask"] == "arithmetic" else d_pred.item() * 2100
    error = abs(pred_val - ex["answer"])
    print(f"{ex['query'][:58]:<60} {pred_val:>8.1f} {ex['answer']:>8.1f} {error:>8.1f}")

In [ ]:
# Cell 7.2 — Batch inference timing
import time

test_queries = [ex["query"] for ex in example_queries] * 20  # 100 queries
test_contexts = [ex["context"] for ex in example_queries] * 20

model.eval()
t0 = time.perf_counter()
for q, c in zip(test_queries, test_contexts):
    text = build_input_text(q, c)
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(device) for k, v in enc.items()}
    ts_norm = torch.tensor([0.5], device=device)
    with torch.no_grad():
        model(enc["input_ids"], enc["attention_mask"], ts_norm)

elapsed = time.perf_counter() - t0
qps = len(test_queries) / elapsed
print(f"{len(test_queries)} queries in {elapsed:.2f}s → {qps:.1f} queries/sec")